# What Makes It Deep

Lecture 2 of 3 on NN/DL.

Last time: neural networks are compositions of things you already know.

Today: what makes *depth* work (activations, backprop), regularization by other names, and the jump from sklearn to PyTorch.

## Setup

NOTE: This notebook takes a while to run. Press run all and let it cook while you are reading. You will still probably end up waiting.

In [ ]:
%matplotlib inline

import time

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# MNIST - same as Lecture 1
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_mnist, y_mnist = mnist.data, mnist.target.astype(int)

X_train, X_test = X_mnist[:60000], X_mnist[60000:]
y_train, y_test = y_mnist[:60000], y_mnist[60000:]

X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0

print(f"Training: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} samples")

---

## Part 1: Activation Functions

### The step → sigmoid → ReLU progression

In Lecture 1 we saw three activation functions in the history of neural networks:

- *Step function* (Perceptron, 1958): hard on/off switch. Not differentiable → can't train deep networks.
- *Sigmoid* (MLP revival, 1986): smooth dimmer switch. Differentiable → but squashes everything to (0, 1), shrinking gradients.
- *ReLU* (modern, 2010s): a differentiable on/off switch. Off when negative, pass-through when positive. No squashing on the positive side → gradients flow at full strength. Simple, fast, works.

To better understand the limitations of Sigmoid, we'll compare it with ReLU and Tanh, another commonly used activation function.

In [ ]:
x = np.linspace(-5, 5, 300)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def tanh_deriv(x):
    return 1 - np.tanh(x)**2

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# Top row: activations
for ax, (name, fn) in zip(axes[0], [
    ("Sigmoid", sigmoid), ("Tanh", np.tanh), ("ReLU", relu)
]):
    ax.plot(x, fn(x), linewidth=2)
    ax.set_title(name)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1.5, 2.5)

# Bottom row: derivatives
for ax, (name, fn) in zip(axes[1], [
    ("Sigmoid derivative (max 0.25)", sigmoid_deriv),
    ("Tanh derivative (max 1.0)", tanh_deriv),
    ("ReLU derivative (0 or 1)", relu_deriv),
]):
    ax.plot(x, fn(x), linewidth=2, color="tomato")
    ax.set_title(name)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.2, 1.5)

plt.suptitle("Activation Functions and Their Derivatives", y=1.02)
plt.tight_layout()
plt.show()

### The vanishing gradient problem

Backpropagation computes gradients by multiplying derivatives layer by layer (chain rule). With sigmoid, each layer's derivative is at most 0.25. After a few layers:

| Layers | Max gradient | Effect |
|---|---|---|
| 2 | $0.25^2 = 0.063$ | Manageable |
| 5 | $0.25^5 = 0.001$ | Early layers barely learn |
| 10 | $0.25^{10} \approx 10^{-6}$ | Effectively zero |

The deeper the network, the less signal reaches the early layers. This is why deep sigmoid networks didn't work for decades - not because the architecture was wrong, but because training couldn't push gradients through many layers.

ReLU's derivative is 1 for positive inputs, so all positive signal passes through without modification.

(ReLU has a kink at exactly zero where it's technically not differentiable. In practice this doesn't matter - the probability of landing exactly on zero is negligible, and frameworks define the gradient as 0 there by convention.)

### Depth comparison: sigmoid vs ReLU

In [ ]:
configs = [
    ("sigmoid, 2 layers", "logistic", (128, 64)),
    ("sigmoid, 4 layers", "logistic", (128, 128, 64, 64)),
    ("relu, 2 layers", "relu", (128, 64)),
    ("relu, 4 layers", "relu", (128, 128, 64, 64)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, act, layers) in zip(axes, configs):
    mlp = MLPClassifier(
        hidden_layer_sizes=layers, activation=act,
        max_iter=50, early_stopping=True, random_state=42,
    )
    mlp.fit(X_train_scaled, y_train)
    ax.plot(mlp.loss_curve_, linewidth=2)
    test_acc = mlp.score(X_test_scaled, y_test)
    ax.set_title(f"{name}\nAcc: {test_acc:.4f}", fontsize=9)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
plt.suptitle("Sigmoid Struggles with Depth; ReLU Does Not", y=1.02)
plt.tight_layout()
plt.show()

With 2 layers, sigmoid and ReLU perform similarly - gradients don't have far to travel. With 4 layers, sigmoid's loss drops more slowly and accuracy suffers.

Note the convergence warning on sigmoid 4-layer: `MLPClassifier` stops when the improvement in loss between iterations falls below `tol` (default $10^{-4}$) or when `max_iter` is reached. The warning means the model was *still improving* at iteration 50 but ran out of time - sigmoid needed more iterations to learn what ReLU learned faster. This is the vanishing gradient problem in action: weaker gradients mean slower convergence. With more layers (8, 10+) the gap becomes dramatic, but even at 4 layers the pattern is clear.

---

## Part 2: Backpropagation Intuition

Backpropagation is the chain rule applied systematically through a computational graph. Not magic - just calculus.

*Forward pass*: inputs flow through layers, each applying weights + activation, until the output layer computes a prediction. Compare the prediction to the label → loss.

*Backward pass*: starting from the loss, compute how much each weight contributed to the error by working backwards through the layers. The chain rule decomposes the total derivative into local derivatives at each step.

The analogy: "the prediction was too high. Backprop traces which weights pushed it up and nudges them down. It's gradient descent applied layer by layer."

The key insight for frameworks: *you only define the forward pass*. The framework computes the backward pass automatically. This is called *autograd* (automatic differentiation), and it's the core technology that makes PyTorch and TensorFlow possible.

A concrete example. Consider the function $y = 3x^2 + 2x + 1$ evaluated at $x = 2$.

*Forward pass* - compute the output step by step:

$$a = x^2 = 4 \quad\to\quad b = 3a = 12 \quad\to\quad c = 2x = 4 \quad\to\quad y = b + c + 1 = 17$$

*Backward pass* - apply the chain rule from output back to input:

$$\frac{dy}{dx} = \frac{dy}{db}\frac{db}{da}\frac{da}{dx} + \frac{dy}{dc}\frac{dc}{dx} = (1)(3)(2x) + (1)(2) = 6x + 2 = 14$$

Each step only requires the *local* derivative at that step. The chain rule links them together. You could compute $dy/dx = 6x + 2$ directly here and get the same answer. The chain rule is valuable because it works even when you can't see the whole function at once - which is the situation in a deep network with millions of parameters. This is exactly what autograd does.

### What is PyTorch?

Before we see autograd in action, a brief introduction.

PyTorch is a framework for building and training neural networks. At its core, it provides two things:

1. *Tensors* - multi-dimensional arrays, like NumPy's `ndarray`, but with two key features: they can live on a GPU (for fast parallel computation), and they track the operations performed on them (for automatic differentiation).

2. *Autograd* - automatic computation of gradients. You define the forward pass (how data flows through the network). PyTorch records every operation and can replay them backwards to compute gradients. No manual calculus required.

The word "tensor" sounds intimidating but it's just the general term for the data structures you already use: a scalar is a 0D tensor, a vector is 1D, a matrix is 2D, and a stack of matrices (like a batch of images) is 3D. NumPy calls them `ndarray`. PyTorch calls them `Tensor`. Same thing, plus GPU support and autograd.

In [ ]:
import torch

# A tensor is just a multi-dimensional array
a = torch.tensor([1.0, 2.0, 3.0])
print(f"1D tensor: {a}")
print(f"Shape: {a.shape}")
print(f"Like numpy: {a.numpy()}")

In [ ]:
b = torch.randn(3, 4)  # random 3x4 matrix
print(f"\n2D tensor (3x4):\n{b}")
print(f"Shape: {b.shape}")

Hardware support is an important consideration for deep learning. PyTorch provides good support for NVIDIA and Apple Silicon, along with CPU for other devices.

In [ ]:
# Check what devices are available
print(f"CPU: always available")
print(f"CUDA (NVIDIA GPU): {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon GPU): {torch.backends.mps.is_available()}")

# Pick the best available device
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"\nUsing: {device}")

On Colab with a GPU runtime, this should say `cuda`. On an Apple Silicon Mac, `mps`. Both give you GPU acceleration. On CPU, everything still works - just slower for large models. Note that on free Colab instances GPU access is limited.

### Autograd: gradients for free

`requires_grad=True` tells PyTorch to record every operation on a variable. `.backward()` walks the recorded operations in reverse, applying the chain rule. `.grad` holds the result.

In this simple demo, `x` plays the role of a weight - it's the variable we want to optimize. In a real network, `requires_grad=True` is set on the *weights and biases* (the parameters being learned), not on the input data. The input data flows through; the weights get updated.

In [ ]:
# Autograd demo: PyTorch computes derivatives automatically
# y = 3x² + 2x + 1, evaluated at x=2
# dy/dx = 6x + 2 = 14 at x=2

x = torch.tensor(2.0, requires_grad=True)  # "track this variable"
y = 3 * x**2 + 2 * x + 1

y.backward()  # compute dy/dx via the chain rule

print(f"y = {y.item():.1f}")
print(f"dy/dx at x=2: {x.grad.item():.1f}  (expected: 14.0)")

This scales to millions of parameters with no change to the code. Same mechanism.

Now the same idea applied to a tiny neural network. We define the weights manually here to show what's happening inside - in practice, `nn.Sequential` handles this for you (as we'll see in Part 4). The weights are initialized randomly, and `requires_grad=True` tells PyTorch to track them for gradient computation.

If all weights start at the same value (e.g., zero), every neuron in a layer computes the same output, receives the same gradient, and gets the same update. They stay identical forever - the network can't learn different features. This is called the symmetry problem. Random initialization breaks that symmetry. Each neuron starts slightly different, so they respond to different patterns in the data and diverge during training.

In [ ]:
# Same idea with a mini neural network
torch.manual_seed(42)

# A tiny network: 3 inputs → 4 hidden neurons → 1 output
W1 = torch.randn(3, 4, requires_grad=True)
b1 = torch.zeros(4, requires_grad=True)
W2 = torch.randn(4, 1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)

The forward pass: data flows through two layers. The input `x` is an arbitrary 3-element vector, and we pick an arbitrary target value of 1.0. The loss measures how far the network's output is from that target using MSE.

In [ ]:
# Forward pass - this is ALL you define
x = torch.tensor([1.0, 2.0, 3.0])
hidden = torch.relu(x @ W1 + b1)            # layer 1: linear + ReLU
output = hidden @ W2 + b2                   # layer 2: linear
loss = (output - torch.tensor([1.0]))**2    # MSE loss

The backward pass: one call computes gradients for every weight in the network.

In [ ]:
# Backward pass - ONE call, all gradients computed
loss.backward()

print(f"Output: {output.item():.4f}")
print(f"Loss:   {loss.item():.4f}")
print(f"W1 gradients: {W1.grad.shape} ({W1.grad.numel()} values)")
print(f"W2 gradients: {W2.grad.shape} ({W2.grad.numel()} values)")

The network predicted -5.8 when the target was 1.0, so the loss is large (-6.8^2 ~ 46). But the important thing is the last two lines: every weight now has a gradient telling it which direction to move to reduce that loss. With random initialization the first prediction is terrible - that's expected. Training is the process of repeating forward pass → backward pass → weight update until the loss is small.

That's the entire mechanism. Forward pass computes the loss. `.backward()` computes all gradients. An optimizer uses the gradients to update the weights. Repeat.

---

## Part 3: Dropout

Regularization concepts from Lecture 1 (weight decay = L2, early stopping = same) all carry over. The one new technique that requires a framework is *dropout*.

Recall from 13a: Random Forests randomly exclude features at each split (`max_features`). This forces each tree to be robust - it can't rely on one dominant feature always being available.

Dropout applies the same idea to neurons. During each training step, each neuron has a probability $p$ of being temporarily "turned off" (set to zero). The network must learn redundant representations - no single neuron can be essential.

In [ ]:
import torch.nn as nn

# Dropout in action
dropout = nn.Dropout(p=0.5)

x = torch.ones(1, 10)

print("Training mode (dropout active):")
dropout.train()
for i in range(3):
    result = dropout(x)
    print(f"  Pass {i+1}: {result.numpy().flatten()}")

print(f"\nEval mode (dropout off):")
dropout.eval()
print(f"  {dropout(x).numpy().flatten()}")

Each pass zeros out different neurons. The network sees a different "thinned" version of itself every training step. During evaluation, all neurons are active (outputs are scaled to compensate for the missing neurons during training).

This is why PyTorch models have `.train()` and `.eval()` modes - dropout (and batch normalization) behave differently during training vs evaluation. You'll see `model.train()` before every training loop and `model.eval()` before every evaluation in the code below. Forgetting to switch modes is one of the most common PyTorch bugs - your model will appear to underperform because dropout is still active during evaluation.

---

## Part 4: From sklearn to PyTorch

### Why leave sklearn?

`MLPClassifier` handles the basics. But it can't do:

- Dropout or batch normalization
- GPU acceleration
- Custom architectures (skip connections, attention)
- Convolutional layers (images) or recurrent layers (sequences)
- Pretrained models

PyTorch gives you all of this and more. The tradeoff: you write the training loop yourself.

### Vocabulary first

Before we write any PyTorch training code, one definition:

A *batch* is a subset of training samples processed together in one gradient update. Instead of computing the loss on all 60,000 samples (slow, memory-intensive) or one sample at a time (noisy, unstable), we use mini-batches - typically 32 to 512 samples. Each batch produces one gradient estimate and one weight update. Larger batches give more stable gradients; smaller batches introduce noise that can help escape local minima. 256 is a common default.

An *epoch* is one complete pass through the entire training set. With 60,000 training samples and a batch size of 256, one epoch is $\lceil 60000 / 256 \rceil = 235$ gradient updates. When sklearn's `MLPClassifier` reports `max_iter=50`, each "iteration" is one epoch.

In Lecture 1, we trained for 50-100 iterations in sklearn. Below we'll train for 20 epochs in PyTorch - roughly the same amount of work, but we'll see the progress at each epoch.

### Side-by-side: same architecture, same result

First, proof that PyTorch does the same thing as sklearn when given the same architecture. We deliberately use identical settings (784 → 128 → 64 → 10, Adam optimizer, same data) so the comparison is apples-to-apples. The numbers 128 and 64 are arbitrary architectural choices - the same ones we used in Lecture 1. The point here isn't to show PyTorch is better; it's to show it's *the same math*.

I'm going to skim over some of the implementation details here. My goal in this lecture is not to teach you everything you need to know about PyTorch, but to demonstrate it in a way that you can build on. The rest is left for a future "Intro to NN/DL Course."

First, recreate the baseline MLP from the previous lecture.

In [ ]:
# sklearn baseline
sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64), max_iter=50,
    early_stopping=True, random_state=42,
)
start = time.time()
sklearn_mlp.fit(X_train_scaled, y_train)
sklearn_time = time.time() - start
sklearn_acc = sklearn_mlp.score(X_test_scaled, y_test)
print(f"sklearn MLPClassifier: {sklearn_acc:.4f} ({sklearn_time:.1f}s, "
      f"{sklearn_mlp.n_iter_} iterations)")

For PyTorch, first convert the data to tensors of appropriate type, and package them in a DataLoader so PyTorch can handle all the details of serving observations to the training / eval systems, including batching and splitting.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors and move to best available device
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# DataLoader handles batching and shuffling
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

Next, define the model architecture step by step and specify the loss metric and optimizer.

In [ ]:
# Define the model - same architecture as sklearn
model = nn.Sequential(
    nn.Linear(784, 128),   # 784 inputs → 128 hidden neurons
    nn.ReLU(),             # activation
    nn.Linear(128, 64),    # 128 → 64 hidden neurons
    nn.ReLU(),             # activation
    nn.Linear(64, 10),     # 64 → 10 output classes
).to(device)

# Loss and optimizer - map to sklearn:
#   CrossEntropyLoss  = log-loss (same as sklearn)
#   Adam, lr=0.001    = solver='adam', learning_rate_init=0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

Then, train the model manually.

In [ ]:
# Training loop - this is what sklearn's .fit() does internally
n_epochs = 20
train_losses = []
test_accs = []

start = time.time()
for epoch in range(n_epochs):
    # --- Training phase ---
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass + weight update
        optimizer.zero_grad()   # clear old gradients
        loss.backward()         # compute new gradients (backprop)
        optimizer.step()        # update weights (gradient descent)

        epoch_loss += loss.item() * len(X_batch)

    avg_loss = epoch_loss / len(train_dataset)
    train_losses.append(avg_loss)

    # --- Evaluation phase ---
    model.eval()
    with torch.no_grad():  # no gradients needed for evaluation
        preds = model(X_test_t.to(device)).argmax(dim=1).cpu()
        acc = (preds == y_test_t).float().mean().item()
    test_accs.append(acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>2d}/{n_epochs}  "
              f"Loss: {avg_loss:.4f}  Test Acc: {acc:.4f}")

pytorch_time = time.time() - start
pytorch_acc = test_accs[-1]

Accuracy increases and loss decreases as training progresses. Until it doesn't. Note what happens between epoch 15 and 20.

Final result:

In [ ]:
print(f"\n{'Method':<30s}  {'Accuracy':>8s}  {'Time':>8s}")
print("-" * 50)
print(f"{'sklearn MLPClassifier':<30s}  {sklearn_acc:>8.4f}  {sklearn_time:>7.1f}s")
print(f"{'PyTorch (' + str(device) + ')':<30s}  {pytorch_acc:>8.4f}  {pytorch_time:>7.1f}s")

Same architecture, similar accuracy. The two frameworks use the same optimizer (Adam) and the same loss function, but differ in details: batch size (sklearn defaults to 200, we used 256), convergence criteria, internal tolerances. These details affect the exact path gradient descent takes, which changes the final weights.

The point isn't that these produce the same number. It's that PyTorch isn't doing anything magical - same math, same ballpark. The reason to use PyTorch is the additional flexibility and features it offers.

### Training curves - same interpretation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# sklearn loss curve
axes[0].plot(sklearn_mlp.loss_curve_, linewidth=2)
axes[0].set_title("sklearn: Training Loss")
axes[0].set_xlabel("Iteration (epoch)")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# PyTorch loss curve
axes[1].plot(train_losses, linewidth=2, color="tomato")
axes[1].set_title("PyTorch: Training Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

# PyTorch test accuracy per epoch
axes[2].plot(test_accs, linewidth=2, color="seagreen")
axes[2].set_title("PyTorch: Test Accuracy per Epoch")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Accuracy")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Same Problem, Same Patterns", y=1.02)
plt.tight_layout()
plt.show()

Same learning curve shape, same interpretation. The sklearn and PyTorch loss curves both drop steeply in the first few epochs as the network moves from random weights to something useful, then flatten as the easy gains are exhausted. The accuracy curve mirrors this - rapid improvement early, then a plateau around 97-98%. These are the same patterns you've seen since the learning rate sensitivity and early stopping plots in Lecture 1.

### Concept mapping

| sklearn | PyTorch | What it does |
|---|---|---|
| `MLPClassifier(hidden_layer_sizes=(128, 64))` | `nn.Sequential(nn.Linear(784,128), nn.ReLU(), ...)` | Define architecture |
| `solver='adam'` | `optim.Adam(model.parameters(), lr=0.001)` | Choose optimizer |
| `loss_curve_` (automatic) | `criterion = nn.CrossEntropyLoss()` | Choose loss function |
| `model.fit(X, y)` | Training loop (forward + backward + step) | Train the model |
| `model.score(X, y)` | `model.eval(); preds.argmax(...)` | Evaluate |
| `max_iter=50` | `n_epochs = 20` | How long to train |
| `early_stopping=True` | Manual: track val loss, save best model | When to stop |
| `alpha=0.0001` (L2) | `weight_decay=0.0001` in optimizer | Regularization |
| Not available | `nn.Dropout(p=0.3)` | Regularization |
| Not available | `nn.BatchNorm1d(128)` | Stabilize training |
| Not available | `.to(device)` for GPU | Hardware acceleration |

The explicit training loop is more code. But you control everything: batch size, what happens each epoch, custom losses, mixed precision, gradient accumulation. That flexibility is why frameworks exist.

### PyTorch dropout and batch normalization

With PyTorch, we can add dropout and batch normalization - two techniques that can improve performance beyond what `MLPClassifier` offers.

In [ ]:
# Enhanced model with dropout + batch normalization
model_enhanced = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),    # normalize activations - like StandardScaler between layers
    nn.Dropout(0.3),        # randomly deactivate 30% of neurons
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.BatchNorm1d(128),
    nn.Dropout(0.3),
    nn.Linear(128, 10),
).to(device)

optimizer_e = optim.Adam(model_enhanced.parameters(), lr=0.001)
enhanced_losses = []
enhanced_accs = []

start = time.time()
for epoch in range(20):
    model_enhanced.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        loss = criterion(model_enhanced(X_batch), y_batch)
        optimizer_e.zero_grad()
        loss.backward()
        optimizer_e.step()
        epoch_loss += loss.item() * len(X_batch)
    enhanced_losses.append(epoch_loss / len(train_dataset))

    model_enhanced.eval()
    with torch.no_grad():
        preds = model_enhanced(X_test_t.to(device)).argmax(1).cpu()
        acc = (preds == y_test_t).float().mean().item()
    enhanced_accs.append(acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>2d}/20  "
              f"Loss: {enhanced_losses[-1]:.4f}  Test Acc: {acc:.4f}")

enhanced_time = time.time() - start
enhanced_acc = enhanced_accs[-1]

print(f"\n{'Method':<40s}  {'Accuracy':>8s}  {'Time':>8s}")
print("-" * 60)
print(f"{'sklearn MLPClassifier (128, 64)':<40s}  {sklearn_acc:>8.4f}  {sklearn_time:>7.1f}s")
print(f"{'PyTorch basic (128, 64)':<40s}  {pytorch_acc:>8.4f}  {pytorch_time:>7.1f}s")
print(f"{'PyTorch + BatchNorm + Dropout (256,128)':<40s}  {enhanced_acc:>8.4f}  {enhanced_time:>7.1f}s")

The enhanced model beats both the sklearn and basic PyTorch versions. The improvement comes from techniques that require a framework: dropout prevents co-adaptation, batch normalization stabilizes training and acts as a mild regularizer.

This is why you leave sklearn for neural networks: not because the same architecture runs faster, but because you gain access to techniques and architectures that sklearn can't express. As we've discussed, when it comes to deep learning, architecture is everything.

In [ ]:
# Accuracy comparison
fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, 21)
ax.plot(epochs, test_accs, linewidth=2, label="PyTorch basic (128, 64)")
ax.plot(epochs, enhanced_accs, linewidth=2, label="PyTorch + BN + Dropout (256, 128)")
ax.axhline(sklearn_acc, color="gray", linestyle="--", label=f"sklearn ({sklearn_acc:.4f})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Test Accuracy")
ax.set_title("The Payoff: Techniques sklearn Can't Do")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The improvement is marginal. Why? Dropout and BatchNorm help with *training dynamics* (preventing overfitting, stabilizing gradients) but they don't change *what the model can see*. All three approaches - sklearn MLP, basic PyTorch, enhanced PyTorch - operate on the same flat 784-element vector. The ceiling isn't the training technique; it's the representation. We'll prove this next.

---

## Part 5: The Limits of Fully Connected Networks

### Fashion-MNIST: a harder image problem

MNIST digits are almost too easy - even simple models do well. Fashion-MNIST (Zalando, 2017) is a drop-in replacement: same 28×28 grayscale images, same 60k/10k split, but clothing items instead of digits. It's harder because t-shirts, pullovers, and coats share more visual similarity than 3s and 8s.

In [ ]:
# Fashion-MNIST - same shape as MNIST, harder problem
fmnist = fetch_openml("Fashion-MNIST", version=1, as_frame=False, parser="auto")
X_fmnist, y_fmnist = fmnist.data, fmnist.target.astype(int)

X_ftrain, X_ftest = X_fmnist[:60000] / 255.0, X_fmnist[60000:] / 255.0
y_ftrain, y_ftest = y_fmnist[:60000], y_fmnist[60000:]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

fig, axes = plt.subplots(1, 10, figsize=(14, 1.5))
for i, ax in enumerate(axes):
    ax.imshow(X_ftrain[i].reshape(28, 28), cmap="gray")
    ax.set_title(class_names[y_ftrain[i]], fontsize=8)
    ax.axis("off")
plt.suptitle("Fashion-MNIST: Same Format, Harder Problem", y=1.1)
plt.tight_layout()
plt.show()

### Tuning sklearn on Fashion-MNIST

We know how to do this - `RandomizedSearchCV` from earlier in the course. Let's find sklearn's best MLP for this problem.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "hidden_layer_sizes": [(128,), (256,), (128, 64), (256, 128), (256, 128, 64)],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.0005, 0.001, 0.002],
}

sklearn_search = RandomizedSearchCV(
    MLPClassifier(max_iter=100, early_stopping=True, random_state=42),
    param_dist,
    n_iter=15,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
)

start = time.time()
sklearn_search.fit(X_ftrain, y_ftrain)
fskl_time = time.time() - start
fskl_acc = sklearn_search.score(X_ftest, y_ftest)

print(f"Best params: {sklearn_search.best_params_}")
print(f"Best CV score: {sklearn_search.best_score_:.4f}")
print(f"Test accuracy: {fskl_acc:.4f}")
print(f"Search time: {fskl_time:.1f}s (15 configs × 3 folds)")

### Tuning PyTorch on Fashion-MNIST

In PyTorch there's no `RandomizedSearchCV`. You write the search loop yourself. More code, but complete control over what you search and how.

Higher-level libraries like Keras, FastAI, PyTorch Lightning, and Skorch (best name ever) provide sklearn-style convenience on top of PyTorch (e.g., `model.fit()`, built-in cross-validation, learning rate finders). We use raw PyTorch here to see what's actually happening under the hood.

In [ ]:
X_ftrain_t = torch.tensor(X_ftrain, dtype=torch.float32)
y_ftrain_t = torch.tensor(y_ftrain, dtype=torch.long)
X_ftest_t = torch.tensor(X_ftest, dtype=torch.float32)
y_ftest_t = torch.tensor(y_ftest, dtype=torch.long)

# Hold out validation set for model selection
X_ftr, X_fval = X_ftrain_t[:54000], X_ftrain_t[54000:]
y_ftr, y_fval = y_ftrain_t[:54000], y_ftrain_t[54000:]

ftrain_loader = DataLoader(
    TensorDataset(X_ftr, y_ftr), batch_size=256, shuffle=True
)
fcriterion = nn.CrossEntropyLoss()

# Configurations to try
configs = [
    {"layers": [256, 128],      "dropout": 0.0, "lr": 0.001, "epochs": 30, "label": "(256,128) no reg"},
    {"layers": [256, 128],      "dropout": 0.1, "lr": 0.001, "epochs": 30, "label": "(256,128) drop=0.1"},
    {"layers": [256, 128],      "dropout": 0.2, "lr": 0.001, "epochs": 30, "label": "(256,128) drop=0.2"},
    {"layers": [512, 256],      "dropout": 0.1, "lr": 0.001, "epochs": 30, "label": "(512,256) drop=0.1"},
    {"layers": [512, 256, 128], "dropout": 0.1, "lr": 0.001, "epochs": 30, "label": "(512,256,128) drop=0.1"},
    {"layers": [256, 128],      "dropout": 0.1, "lr": 0.0005, "epochs": 40, "label": "(256,128) drop=0.1 lr=5e-4"},
]

print(f"{'Config':<35s}  {'Val Acc':>8s}")
print("-" * 47)

best_val_acc = 0
best_config = None

for cfg in configs:
    # Build model dynamically
    layers = []
    prev = 784
    for h in cfg["layers"]:
        layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h)])
        if cfg["dropout"] > 0:
            layers.append(nn.Dropout(cfg["dropout"]))
        prev = h
    layers.append(nn.Linear(prev, 10))

    model = nn.Sequential(*layers).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg["lr"])

    for epoch in range(cfg["epochs"]):
        model.train()
        for Xb, yb in ftrain_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = fcriterion(model(Xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        val_acc = (model(X_fval.to(device)).argmax(1).cpu() == y_fval).float().mean().item()

    print(f"{cfg['label']:<35s}  {val_acc:>8.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_config = cfg
        best_model = model

print(f"\nBest: {best_config['label']} (val acc: {best_val_acc:.4f})")

For comparison, here's what the same model looks like in `skorch`, which wraps PyTorch in the sklearn API:

```python
# Not executable - just for illustration
from skorch import NeuralNetClassifier

net = NeuralNetClassifier(
    module=nn.Sequential(
        nn.Linear(784, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.1),
        nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.1),
        nn.Linear(128, 10),
    ),
    max_epochs=30,
    lr=0.001,
    optimizer=optim.Adam,
    criterion=nn.CrossEntropyLoss,
    device=device,
)

# Now it works like any sklearn estimator
net.fit(X_train, y_train)
net.score(X_test, y_test)
cross_val_score(net, X, y, cv=5)  # even this works
```

Same PyTorch model, same training, but the loop is handled for you. If you want to use a neural network in your project without writing a manual training loop, `skorch` is a good option.

Now that we've identified the best model (hyperparameters, including architecture), we use that to get the best fit (all data):

In [ ]:
# Retrain best config on full training set for fair comparison
layers = []
prev = 784
for h in best_config["layers"]:
    layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h)])
    if best_config["dropout"] > 0:
        layers.append(nn.Dropout(best_config["dropout"]))
    prev = h
layers.append(nn.Linear(prev, 10))

fmodel = nn.Sequential(*layers).to(device)
foptimizer = optim.Adam(fmodel.parameters(), lr=best_config["lr"])

full_loader = DataLoader(
    TensorDataset(X_ftrain_t, y_ftrain_t), batch_size=256, shuffle=True
)

start = time.time()
for epoch in range(best_config["epochs"]):
    fmodel.train()
    for Xb, yb in full_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        loss = fcriterion(fmodel(Xb), yb)
        foptimizer.zero_grad()
        loss.backward()
        foptimizer.step()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        fmodel.eval()
        with torch.no_grad():
            preds = fmodel(X_ftest_t.to(device)).argmax(1).cpu()
            acc = (preds == y_ftest_t).float().mean().item()
        print(f"Epoch {epoch+1:>2d}/{best_config['epochs']}  Test Acc: {acc:.4f}")

fpt_time = time.time() - start
fmodel.eval()
with torch.no_grad():
    preds = fmodel(X_ftest_t.to(device)).argmax(1).cpu()
    fpt_acc = (preds == y_ftest_t).float().mean().item()

print(f"\n{'Method':<45s}  {'Test Acc':>8s}  {'Time':>8s}")
print("-" * 65)
print(f"{'sklearn (tuned via RandomizedSearchCV)':<45s}  {fskl_acc:>8.4f}  {fskl_time:>7.1f}s")
print(f"{'PyTorch (tuned, BN + Dropout)':<45s}  {fpt_acc:>8.4f}  {fpt_time:>7.1f}s")

Both frameworks, properly tuned, land in the same neighborhood: roughly 89%. Adding layers, dropout, batch normalization, tuning the learning rate. Lots of work for limited / negative returns!

The ceiling isn't the framework or the training tricks. It's the *architecture*. An MLP flattens a 28×28 image into a 784-element vector and treats every pixel as independent. It has no concept of spatial structure.

This is the most important result in this notebook: *the tools don't matter if the architecture is wrong for the data*.

### What does it get wrong?

In [ ]:
fmodel.eval()
with torch.no_grad():
    preds = fmodel(X_ftest_t.to(device)).argmax(1).cpu().numpy()

wrong = np.where(preds != y_ftest)[0]
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for i, ax in enumerate(axes.ravel()):
    idx = wrong[i]
    ax.imshow(X_ftest[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"True: {class_names[y_ftest[idx]]}\n"
                 f"Pred: {class_names[preds[idx]]}", fontsize=8)
    ax.axis("off")
plt.suptitle("MLP Mistakes on Fashion-MNIST", y=1.02)
plt.tight_layout()
plt.show()

The MLP confuses visually similar items: shirts vs pullovers, coats vs dresses. Even with BatchNorm and Dropout, the fundamental limitation remains: an MLP treats the image as a flat vector of 784 independent pixels. It has no concept of *spatial structure* - it doesn't know that pixel (5, 5) is next to pixel (5, 6).

### The representation problem

To see why spatial structure matters, look at the same image two ways:

In [ ]:
idx = 0
img = X_ftest[idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 3),
                         gridspec_kw={"width_ratios": [1, 4]})

# 28x28 grid - what a human sees
axes[0].imshow(img.reshape(28, 28), cmap="gray")
axes[0].set_title(f"28×28 image\n({class_names[y_ftest[idx]]})", fontsize=10)
axes[0].axis("off")

# 784x1 strip - what an MLP sees
axes[1].imshow(img.reshape(1, 784), cmap="gray", aspect="auto")
axes[1].set_title("784-element flat vector (same pixels, no spatial structure)", fontsize=10)
axes[1].set_yticks([])
axes[1].set_xlabel("Pixel index")

plt.suptitle("Same Data, Different Representation", y=1.05)
plt.tight_layout()
plt.show()

Which would you rather classify from? The image on the left preserves spatial relationships - you can see a shape. The strip on the right has the exact same pixel values but the spatial structure is destroyed. An MLP (and an SVM on flat pixels) sees the strip. A model with spatial awareness sees the grid.

Let's prove this matters with numbers.

### Step 1: A better model doesn't help

SVM with an RBF kernel is one of the most powerful classifiers we've used in this course. But it still operates on the flat 784-element vector.

In [ ]:
from sklearn.svm import SVC

# SVM on flat pixels - subsample for speed
np.random.seed(42)
svm_idx = np.random.choice(len(X_ftrain), 10000, replace=False)

svm_flat = SVC(kernel="rbf", random_state=42)
start = time.time()
svm_flat.fit(X_ftrain[svm_idx], y_ftrain[svm_idx])
svm_flat_time = time.time() - start
svm_flat_acc = svm_flat.score(X_ftest, y_ftest)

print(f"SVM (rbf) on flat pixels: {svm_flat_acc:.4f} ({svm_flat_time:.1f}s, "
      f"trained on {len(svm_idx):,} samples)")

Same ceiling. The RBF kernel measures similarity in pixel-intensity space, but it has no concept of spatial adjacency. No standard SVM kernel does - they all treat the input as a flat vector. A better model on a bad representation can't overcome the representation.

### Step 2: A better representation helps

*Histogram of Oriented Gradients* (HOG) is a hand-engineered feature extractor for images. It divides the image into small patches and counts the dominant edge directions in each patch. The output is a feature vector that encodes *local spatial structure* - where edges are and which direction they point.

In [ ]:
from skimage.feature import hog

# Compute HOG features for one image to visualize
img_2d = X_ftest[0].reshape(28, 28)
hog_features, hog_image = hog(
    img_2d, orientations=9, pixels_per_cell=(4, 4),
    cells_per_block=(2, 2), visualize=True,
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(img_2d, cmap="gray")
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(hog_image, cmap="gray")
axes[1].set_title("HOG visualization\n(edge orientations per patch)")
axes[1].axis("off")

axes[2].bar(range(len(hog_features)), hog_features, width=1.0, color="steelblue")
axes[2].set_title(f"HOG feature vector ({len(hog_features)} features)")
axes[2].set_xlabel("Feature index")
axes[2].set_ylabel("Magnitude")

plt.suptitle("HOG: Hand-Engineered Spatial Features", y=1.05)
plt.tight_layout()
plt.show()

print(f"Flat pixels: {784} features (no spatial info)")
print(f"HOG features: {len(hog_features)} features (encodes local edge structure)")

In [ ]:
# Extract HOG features for all images
X_ftrain_hog = np.array([
    hog(x.reshape(28, 28), orientations=9, pixels_per_cell=(4, 4),
        cells_per_block=(2, 2))
    for x in X_ftrain
])
X_ftest_hog = np.array([
    hog(x.reshape(28, 28), orientations=9, pixels_per_cell=(4, 4),
        cells_per_block=(2, 2))
    for x in X_ftest
])

# SVM on HOG features - same subsample for fair comparison
svm_hog = SVC(kernel="rbf", random_state=42)
start = time.time()
svm_hog.fit(X_ftrain_hog[svm_idx], y_ftrain[svm_idx])
svm_hog_time = time.time() - start
svm_hog_acc = svm_hog.score(X_ftest_hog, y_ftest)

print(f"SVM (rbf) on HOG features: {svm_hog_acc:.4f} ({svm_hog_time:.1f}s)")

Same model (SVM rbf), different representation. The improvement comes entirely from giving the model features that preserve spatial structure. But HOG is hand-engineered - someone decided "4×4 patches, 9 orientation bins." What if the network could learn its own spatial features?

### Step 3: A learned representation is even better

A CNN learns its own spatial features during training. Instead of you specifying "count edge orientations in 4×4 patches" (HOG), the network discovers which local patterns matter through backpropagation. Same principle as hidden layers learning features in Lecture 1, but now with spatial awareness built into the architecture.

In [ ]:
# Minimal CNN - 2 convolutional layers + 1 fully connected
cnn = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),    # 28×28×1 → 28×28×32
    nn.ReLU(),
    nn.MaxPool2d(2),                               # → 14×14×32
    nn.Conv2d(32, 64, kernel_size=3, padding=1),   # → 14×14×64
    nn.ReLU(),
    nn.MaxPool2d(2),                               # → 7×7×64
    nn.Flatten(),                                  # → 3136
    nn.Linear(7 * 7 * 64, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10),
).to(device)

# Reshape data to 2D images (batch, channels, height, width)
X_ftrain_2d = torch.tensor(X_ftrain, dtype=torch.float32).reshape(-1, 1, 28, 28)
X_ftest_2d = torch.tensor(X_ftest, dtype=torch.float32).reshape(-1, 1, 28, 28)
y_ftrain_t2 = torch.tensor(y_ftrain, dtype=torch.long)
y_ftest_t2 = torch.tensor(y_ftest, dtype=torch.long)

cnn_loader = DataLoader(
    TensorDataset(X_ftrain_2d, y_ftrain_t2), batch_size=256, shuffle=True
)
cnn_optimizer = optim.Adam(cnn.parameters(), lr=0.001)
cnn_criterion = nn.CrossEntropyLoss()

start = time.time()
for epoch in range(15):
    cnn.train()
    for Xb, yb in cnn_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        loss = cnn_criterion(cnn(Xb), yb)
        cnn_optimizer.zero_grad()
        loss.backward()
        cnn_optimizer.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        cnn.eval()
        with torch.no_grad():
            preds = cnn(X_ftest_2d.to(device)).argmax(1).cpu()
            acc = (preds == y_ftest_t2).float().mean().item()
        print(f"Epoch {epoch+1:>2d}/15  Test Acc: {acc:.4f}")

cnn_time = time.time() - start
cnn.eval()
with torch.no_grad():
    cnn_acc = (cnn(X_ftest_2d.to(device)).argmax(1).cpu() == y_ftest_t2).float().mean().item()

In [ ]:
print(f"\n{'Approach':<40s}  {'Representation':<25s}  {'Test Acc':>8s}")
print("-" * 78)
print(f"{'MLP (tuned, BN + Dropout)':<40s}  {'flat pixels (784)':<25s}  {fpt_acc:>8.4f}")
print(f"{'SVM rbf (flat pixels)':<40s}  {'flat pixels (784)':<25s}  {svm_flat_acc:>8.4f}")
print(f"{'SVM rbf (HOG features)':<40s}  {'hand-engineered spatial':<25s}  {svm_hog_acc:>8.4f}")
print(f"{'CNN (2 conv layers)':<40s}  {'learned spatial':<25s}  {cnn_acc:>8.4f}")

The story these numbers tell:

- MLP vs SVM on flat pixels: both land in the same neighborhood (SVM was trained on a 10k subsample for speed, so its lower number reflects less training data, not a worse model). Neither breaks through the ceiling that flat pixels impose. The representation is the bottleneck, not the model.
- SVM (flat) → SVM (HOG): the same model on a better representation *does* help. HOG preserves spatial structure that flat pixels destroy.
- SVM (HOG) → CNN: a model that *learns* its own spatial representation beats hand-engineered features. The CNN discovers what HOG was designed to capture - and more.

This is the whole story of deep learning in one table: *better representations beat better models, and learned representations beat hand-engineered ones*.

### Architecture as inductive bias

The transition from "which framework?" to "which architecture?" is the transition that matters:

| Architecture | Inductive Bias | Good For |
|---|---|---|
| Fully connected (MLP) | None - treats all inputs equally | Tabular data |
| CNN (Convolutional) | Local spatial patterns, translation invariance | Images |
| RNN / LSTM | Sequential order matters | Time series, text (older) |
| Transformer | Relevance depends on content, not position | Text, everything (modern) |

These aren't magic. They're assumptions about data structure baked into the architecture. CNNs assume nearby pixels relate to each other. Transformers assume that which parts of the input matter depends on what the input *says*, not where it appears.

Architecture is the one design lever that matters at scale. And it's developed empirically - intuition and experiment, not theorems. That's the subject of Lecture 3.

---

## Summary

| Lecture 1 | Lecture 2 |
|---|---|
| Neuron = logistic regression | Depth requires non-linear activations |
| Hidden layer = learned features | ReLU: a differentiable on/off switch |
| `MLPClassifier` API | Backprop = chain rule; autograd computes it for you |
| Regularization has NN names | PyTorch: same math, you control the loop |
| | Dropout + BatchNorm: the payoff for leaving sklearn |
| | Representation > model; learned > hand-engineered |